In [ ]:

# Step 1: Import Libraries

import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense


# Step 2: Load Dataset

data = pd.read_csv("./LP IV Updated Datasets/CBOW/IMDB Dataset.csv")

print(data.head())

# Step 3: Preprocessing Data

data["review"] = data["review"].str.lower()

data["review"] = data["review"].apply(
    lambda x: re.sub(r'[^a-z\s]', '', x)
)

# Convert reviews into words

sentences = data["review"].apply(lambda x: x.split()).tolist()

# Convert sentiment into numbers

data["sentiment"] = data["sentiment"].map({
    "positive": 1,
    "negative": 0
})

y = data["sentiment"].values

# Step 4: Train CBOW Word2Vec Model

cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=5,
    min_count=2,
    sg=0,
    workers=4
)

# Step 5: Convert Documents into Embeddings

def document_vector(doc):
    words = [w for w in doc if w in cbow_model.wv]

    if len(words) == 0:
        return np.zeros(50)

    return np.mean(cbow_model.wv[words], axis=0)

X = np.array([
    document_vector(doc) for doc in sentences
])

# Step 6: Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 7: Build Classification Model

model = Sequential([
    Dense(64, activation="relu", input_shape=(50,)),
    Dense(1, activation="sigmoid")
])

# Step 8: Compile Model

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Step 9: Train Model

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# Step 10: Evaluate Model

loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)
